# AI as Judge

[G-Eval](https://deepeval.com/docs/metrics-llm-evals) is a framework that uses LLM as a judge to evaluate LLM outputs. The evaluation can be based on any criteria. G-Eval is implemented by a library called [DeepEval](https://deepeval.com/) which includes a broader set of tests.


In [17]:
%load_ext dotenv
%dotenv ../../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [18]:
from openai import OpenAI
import os

document_folder = "../../05_src/documents/"
blue_cross_file = "chesterton.txt"
file_path = os.path.join(document_folder, blue_cross_file)

with open(file_path, "r", encoding="utf-8") as f:
    blue_cross_text = f.read()

In [19]:
instructions = "You are an helpful assistant that summarizes works of fiction with a quirky and bubbly approach."
PROMPT = """
    Summarize the following story in at most four paragraphs. Please include all key characters and plot points.
    <story>
    {story}
    </story>
    In addition to the summary, add an introduction paragraph where you greet the reader and a conclusion where you share an opinion about the story.
"""

In [20]:
client = OpenAI()
response = client.responses.create(
    model="gpt-4o-mini",
    instructions=instructions,
    input=[
        {"role": "user", 
         "content": PROMPT.format(story=blue_cross_text)}
    ],
    temperature=1.2
)

In [21]:
response.output_text

'**Hello there, delightful reader! 🌟** If you’re ready to dive into a world brimming with ingenious mysteries, peculiar characters, and delightful wit, you\'re in for a treat! Let’s take a whimsical journey through the crafty tales of Father Brown as he uncovers various mysteries in *The Innocence of Father Brown* by G.K. Chesterton.\n\nThe stories within this charming collection revolve around Father Brown, a seemingly humble Roman Catholic priest who embodies wisdom and intellect beyond his small stature. Alongside his towering friend, Flambeau, he navigates peculiar cases that unravel essential truths about humanity. From tracking a notorious criminal, Flambeau, in "The Blue Cross," to confronting grisly murders and ethical dilemmas, Brown\'s deductions reveal how empathy and understanding can shine light on seemingly dark paths. Every interaction is peppered with witty banter and unexpected insights, making it clear that appearances can be extremely deceiving!\n\nAs the narratives 

# Answer Relevancy

The answer relevancy metric evaluates how relevant the actual output of the LLM app is compared to the provided input. This metric is self-explaining in the sense that the output includes a reason for the metric score.

The metric is calculated as:

$$
AnswerRelevancy=\frac{NumberRelevantStatements}{TotalStatements}
$$

Reference: [Answer Relevancy](https://deepeval.com/docs/metrics-answer-relevancy). 

In [22]:
from deepeval import evaluate
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase

metric = AnswerRelevancyMetric(
    threshold=0.7,
    model="gpt-4o-mini",
    include_reason=True
)

test_case = LLMTestCase(
    input=PROMPT.format(story=blue_cross_text),
    actual_output=response.output_text
)

In [23]:
metric.measure(test_case)

Output()

1.0

In [24]:
print(metric.score,metric.reason)

1.0 The score is 1.00 because the response directly addresses the request for a summary of the story, including key characters and plot points, without any irrelevant statements or digressions.


# Other Metrics

Other useful metric functions include:

+ [Faithfulness](https://deepeval.com/docs/metrics-faithfulness): evaluates whether the `actual_output` factually aligns with the contents of  `retrieval_context`. 
+ [Contextual Precision](https://deepeval.com/docs/metrics-contextual-precision): evaluates whether nodes in your `retrieval_context` that are relevant to the given input are ranked higher than irrelevant ones. 
+ [Contextual Recall](https://deepeval.com/docs/metrics-contextual-recall): evaluates the extent of which the retrieval_context aligns with the expected_output. 
+ [Contextual Relevancy](https://deepeval.com/docs/metrics-contextual-relevancy): evaluates the overall relevance of the information presented in your retrieval_context for a given input. 

# G-Eval

[G-Eval](https://deepeval.com/docs/metrics-llm-evals) is a framework that uses LLM-as-a-judge with chain-of-thoughts (CoT) to evaluate LLM outputs based on ANY custom criteria. The G-Eval metric is the most versatile type of metric deepeval offers.

In [25]:
instructions = "You are an helpful assistant that specializes in works of fiction."
PROMPT = """
    Based on the story below, answer the question provided.
    <story>
    {story}
    </story>
    <question>
    Who is the main antagonist in the story and what motivates their actions?
    </question>
"""

In [26]:
client = OpenAI()
response = client.responses.create(
    model="gpt-4o-mini",
    instructions=instructions,
    input=[
        {"role": "user", 
         "content": PROMPT.format(story=blue_cross_text)}
    ],
    temperature=0.7
)

In [27]:
response.output_text

'In "The Innocence of Father Brown," the main antagonist can be seen as **Flambeau**, though his character evolves throughout the stories. Initially depicted as a notorious criminal, his motivations stem from a desire for excitement, challenge, and ultimately, a search for redemption.\n\nThroughout the collection, Flambeau engages in various criminal activities, driven by his cleverness and artistry in crime. However, as he encounters Father Brown, who represents morality and wisdom, Flambeau\'s character begins to shift. He becomes intrigued by Father Brown\'s insights and eventually seeks to reform, revealing a conflict between his past criminal instincts and a newfound respect for justice.\n\nIn summary, Flambeau serves as the antagonist primarily through his criminal actions, motivated by thrill and later by the desire for a more noble purpose in life. His relationship with Father Brown highlights themes of redemption, morality, and the complexity of human nature.'

## Evaluation Criteria

The most straightforward way to establish a metric is by using a single criteria.

In [28]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

correctness_metric = GEval(
    name="Correctness",
    criteria="Determine whether the actual output is factually correct based on the context.",
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
)

In [29]:
test_case = LLMTestCase(
    input=PROMPT.format(story=blue_cross_text),
    actual_output=response.output_text
)
evaluate(test_cases=[test_case], metrics=[correctness_metric])

✨ You're running DeepEval's latest Correctness [GEval] Metric! (using gpt-4.1, strict=False, async_mode=True)...

Output()

PermissionDeniedError: Error code: 403 - {'error': {'message': 'Project `proj_azcDlGrYmDy6eV8yO8hoT2pv` does not have access to model `gpt-4.1`', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}}

## Evaluation Steps 

G-Eval is flexible in many ways: notice that we can establish an evaluation criteria or a set of evaluation steps, that can help in guiding the model to follow specific steps to perform the evaluation.

In [ ]:
...

correctness_metric = GEval(
    name="Correctness",
    evaluation_steps=[
        "Check whether the facts in 'actual output' contradicts any facts in 'input'",
        "You should also heavily penalize omission of detail",
        "Vague language, or contradicting OPINIONS, are not OK"
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
)

In [ ]:
test_case = LLMTestCase(
    input=PROMPT.format(story=blue_cross_text),
    actual_output=response.output_text
)
evaluate(test_cases=[test_case], metrics=[correctness_metric])